In [39]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install opencv-contrib-python

  Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl (45.5 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import cv2
from ultralytics import YOLO
import torch
import numpy as np
from PIL import Image  # Import the Image class from the PIL library

# Load your trained YOLOv8 model
model = YOLO(r"C:\Users\Malakit Ram\ultralytics\models\V11\Classification\yolo11s-cls.pt")  # use your custom model

def classify_camera():
    # Open the camera feed
    cap = cv2.VideoCapture(0)  # 0 for default camera, change index for external cameras

    while True:
        # Capture frame-by-frame
        ret, frame = cap.read()

        if not ret:
            print("Failed to capture image")
            break

        # Resize the frame to 64x64 as required by the model for prediction
        resized_frame = cv2.resize(frame, (64, 64))

        # Convert the OpenCV frame (NumPy array) to a PIL image
        pil_image = Image.fromarray(cv2.cvtColor(resized_frame, cv2.COLOR_BGR2RGB))

        # Convert PIL image to a tensor (required for YOLO models)
        pil_image_tensor = torch.from_numpy(np.array(pil_image)).permute(2, 0, 1).float() / 255.0
        pil_image_tensor = pil_image_tensor.unsqueeze(0)  # Add a batch dimension

        # Predict with the model (passing the tensor image)
        results = model.predict(pil_image_tensor)

        # Get the class probabilities
        probs = results[0].probs  # This gives the class probabilities

        # Get the top 5 class predictions and their confidence scores
        top5_ids = probs.top5  # Indices of the top 5 classes
        top5_conf = probs.top5conf  # Confidence scores of the top 5 classes

        # Get the class names from the model
        class_names = model.names  # This contains the class labels

        # Prepare a list of formatted predictions to display
        predictions_list = []
        for i in range(5):
            class_name = class_names[top5_ids[i]]
            confidence = top5_conf[i]
            predictions_list.append(f'{class_name}: {confidence:.2f}')

        # Define highlight colors for each classification (BGR format)
        highlight_colors = [(0, 255, 0),  # Green for the top prediction
                            (0, 255, 255),  # Yellow
                            (173, 216, 230),  # Light Blue
                            (0, 165, 255),  # Orange
                            (255, 0, 255)]  # Magenta

        # Calculate font size and thickness dynamically based on image dimensions
        h, w, _ = frame.shape
        font_scale = max(w, h) / 1000.0  # Adjust font scale proportionally to the image size
        font_thickness = max(1, int(max(w, h) / 500))  # Adjust thickness based on the image size

        # Display the predictions list on the frame inside rectangles
        y_offset = int(h * 0.1)  # Start below the top of the frame
        for i, prediction in enumerate(predictions_list):
            color = highlight_colors[i % len(highlight_colors)]  # Cycle through the defined colors

            # Calculate text size and box size
            (text_width, text_height), baseline = cv2.getTextSize(prediction, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)
            box_coords = ((10, y_offset + i * (text_height + 15)), (10 + text_width, y_offset + i * (text_height + 15) + text_height + baseline))

            # Draw the rectangle (filled) for the prediction
            cv2.rectangle(frame, box_coords[0], box_coords[1], color, cv2.FILLED)

            # Put the text prediction on top of the rectangle
            cv2.putText(frame, prediction, (10, y_offset + i * (text_height + 15) + text_height), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), font_thickness, cv2.LINE_AA)

        # Show the resulting frame
        cv2.imshow('Camera Feed', frame)

        # Break the loop if the user presses the 'q' key
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the camera and close any OpenCV windows
    cap.release()
    cv2.destroyAllWindows()

# Run the classify camera function
classify_camera()



0: 64x64 wall_clock 0.13, hair_spray 0.13, nipple 0.07, croquet_ball 0.04, hook 0.03, 163.8ms
Speed: 1.2ms preprocess, 163.8ms inference, 2.6ms postprocess per image at shape (1, 3, 64, 64)

0: 64x64 wall_clock 0.13, hair_spray 0.13, nipple 0.09, croquet_ball 0.03, envelope 0.02, 32.4ms
Speed: 0.0ms preprocess, 32.4ms inference, 0.3ms postprocess per image at shape (1, 3, 64, 64)

0: 64x64 wall_clock 0.14, hair_spray 0.13, nipple 0.09, croquet_ball 0.03, envelope 0.02, 37.0ms
Speed: 0.0ms preprocess, 37.0ms inference, 0.2ms postprocess per image at shape (1, 3, 64, 64)

0: 64x64 wall_clock 0.14, hair_spray 0.13, nipple 0.09, croquet_ball 0.03, hook 0.03, 39.2ms
Speed: 0.0ms preprocess, 39.2ms inference, 0.5ms postprocess per image at shape (1, 3, 64, 64)

0: 64x64 wall_clock 0.14, hair_spray 0.11, nipple 0.11, croquet_ball 0.03, hook 0.03, 19.1ms
Speed: 0.0ms preprocess, 19.1ms inference, 0.3ms postprocess per image at shape (1, 3, 64, 64)

0: 64x64 nipple 0.12, wall_clock 0.10, hair_